<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/Copy_of_Langchain_chains_agent_15thov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#secret Key

import os
from google.colab import userdata

# Retrieve API keys from Colab's secure storage

openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

# Langchain setup

In [ ]:
!pip install openai==0.28.1 langchain==0.0.270 langchain-community

In [ ]:
#secret Key

import os
from google.colab import userdata

# Retrieve API keys from Colab's secure storage

openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

## Chains

Restaraunt Business Generator

In [ ]:
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains.sequential import SequentialChain
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Use ChatOpenAI with a chat-optimized model like gpt-3.5-turbo
llm = ChatOpenAI(temperature = 1, model_name="gpt-3.5-turbo")

prompt_template_name= PromptTemplate(
    input_variables = ['cuisine'],
    template = "i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
                                         )


name_chain=LLMChain(llm=llm,prompt=prompt_template_name,output_key="restaraunt_name")




# Re-initialize llm with ChatOpenAI for the second chain as well
llm = ChatOpenAI(temperature = 1, model_name="gpt-3.5-turbo")
prompt_template_items = PromptTemplate(input_variables=['restaraunt_name', 'cuisine'],
                                      template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names ,break down according to category like drinks soup starters main course"
                                       )

food_items_chain=LLMChain(llm=llm,prompt=prompt_template_items,output_key="menu_items")




chain = SequentialChain(
    chains = [name_chain, food_items_chain],
    input_variables =['cuisine'],
    output_variables = ['restaraunt_name','menu_items']
)


In [ ]:
chain({"cuisine": "Pakistani"})

{'cuisine': 'Pakistani',
 'restaraunt_name': '"Saffron Delights"',
 'menu_items': 'Drinks:\n1. Mango Lassi\n2. Kashmiri Chai (Pink Tea)\n3. Sugarcane Juice\n\nSoup:\n1. Nihari\n2. Paya Soup\n\nStarters:\n1. Seekh Kebabs\n2. Pakoras (Assorted Vegetable Fritters)\n3. Samosas\n4. Chicken Reshmi Kebabs\n\nMain Course:\n1. Biryani (Chicken or Lamb)\n2. Karahi Gosht (Spicy Mutton Curry)\n3. Chicken Handi\n4. Haleem\n5. Palak Paneer (Spinach and Cottage Cheese)\n6. Daal Makhani (Black Lentils)\n7. Chicken Tikka Masala\n8. Sindhi Biryani'}

In [ ]:
chain({"cuisine": "Italian"})

{'cuisine': 'Italian',
 'restaraunt_name': '"Bella Cucina Italiana"',
 'menu_items': 'Drinks:\n1. Limoncello Spritz\n2. Italian Margarita\n3. Aperol Spritz\n4. Italian red or white wine\n5. Italian soda\n\nSoup:\n1. Minestrone soup\n2. Zuppa Toscana\n3. Stracciatella soup\n4. Pasta e Fagioli\n5. Cioppino\n\nStarters:\n1. Caprese salad\n2. Bruschetta al Pomodoro\n3. Arancini\n4. Calamari Fritti\n5. Panzerotti\n\nMain Course:\n1. Spaghetti Carbonara\n2. Risotto alla Milanese\n3. Osso Buco\n4. Eggplant Parmigiana\n5. Fettuccine Alfredo\n\n(Depending on the menu items available at Bella Cucina Italiana)'}

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name', 'cuisine'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names ,break down according to category like drinks soup starters main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo")
            gr.Markdown("**Restaurant Business Generator**\nChoose a cuisine type and generate a restaurant name and menu instantly.")
            cuisine_input = gr.Textbox(label="Cuisine Type", placeholder="e.g. Italian, Japanese, Mexican")
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=generate_restaurant, inputs=cuisine_input, outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()

/tmp/ipython-input-755764266.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4f403bcd003f4ff5d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:


import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name', 'cuisine'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names, break down according to category like drinks, soup, starters, main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Popular cuisines dropdown options
TOP_CUISINES = [
    "Italian", "Chinese", "Japanese", "Mexican", "Indian",
    "French", "Thai", "Mediterranean", "American", "Spanish"
]

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo")
            gr.Markdown(
                "**Restaurant Business Generator**  \n"
                "Choose a cuisine type, or select from popular options, then generate a restaurant name and menu instantly."
            )
            cuisine_dropdown = gr.Dropdown(choices=TOP_CUISINES, label="Popular Cuisines", value=TOP_CUISINES[0])
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=generate_restaurant, inputs=cuisine_dropdown, outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()


/tmp/ipython-input-786292604.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://399c931110eb8c240f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name', 'cuisine'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names, break down according to category like drinks, soup, starters, main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Popular cuisines dropdown options
TOP_CUISINES = [
    "Italian", "Chinese", "Japanese", "Mexican", "Indian",
    "French", "Thai", "Mediterranean", "American", "Spanish"
]

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Wrapper to choose between dropdown and custom input
def on_generate(selected, custom):
    cuisine = custom.strip() if custom and custom.strip() else selected
    return generate_restaurant(cuisine)

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    # Center-aligned app title
    gr.Markdown("<h1 style='text-align:center'>Restaurant Business Generator</h1>")
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            # Logo resized to 300px width
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo", width=300)
            gr.Markdown(
                "Choose a cuisine from popular options or type your own, then generate a restaurant name and menu instantly."
            )
            cuisine_dropdown = gr.Dropdown(choices=TOP_CUISINES, label="Popular Cuisines", value=TOP_CUISINES[0])
            cuisine_textbox = gr.Textbox(label="Or enter custom cuisine", placeholder="e.g. Vietnamese, Peruvian")
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=on_generate, inputs=[cuisine_dropdown, cuisine_textbox], outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()


/tmp/ipython-input-1073315120.py:47: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38588c18c1b39b0a3a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Agents

In [ ]:
import os
from google.colab import userdata

SERPAPI_API_KEY = userdata.get("SERP_API")

# Set them as environment variables

if SERPAPI_API_KEY:
    os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY

In [ ]:
pip install google-search-results

#### serpapi and llm-math tool

In [ ]:
pip install wikipedia

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Initialize LLM with a supported chat model
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["serpapi","wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Let's test it out!
agent.run("What was the GDP of US in 2024? Add 5 trillion to this number.")



> Entering new AgentExecutor chain...
I need to find the GDP of the US in 2024 first before adding 5 trillion to it.
Action: Search
Action Input: "US GDP 2024"
Observation: Real GDP increased 2.8 percent in 2024 (from the 2023 annual level to the 2024 annual level), the same as previously estimated. The increase in ...
Thought:I need to find the actual GDP number in 2024.
Action: Search
Action Input: "US GDP 2024 in trillions"
Observation: Global gross domestic product amounted to approximately 117.2 trillion U.S. dollars in 2025, with the United States making up more than one-quarter of this ...
Thought:I need to keep looking for the specific GDP number for the US in 2024.
Action: Search
Action Input: "US GDP in trillions 2024"
Observation: Current-dollar nominal GDP increased by 6.04 percent annualized, or $443.616 billion, to a level of $30.486 trillion, the current size of the U.S. economy.
Thought:I now have the GDP of the US in 2024, which is $30.486 trillion. Now I need to add

'$35.486 trillion'

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Initialize LLM with a supported chat model
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["serpapi","wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Let's test it out!
#agent.run("what is the per capita income of Dubai? Use a search engine to find #the most recent data..")

agent.run("When was Elon musk born? What is his current age in days?")



> Entering new AgentExecutor chain...
I need to find out Elon Musk's birthdate first before calculating his current age in days.
Action: Wikipedia
Action Input: Elon Musk
Observation: Page: Elon Musk
Summary: Elon Reeve Musk (born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the world since 2021; as of October 2025, Forbes estimates his net worth to be around $500 billion.
Born into a wealthy family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania in Philadelphia, United States, before moving to California to pursue business ventures. In 1995, Musk co-founded the software company Zip2. Following its sale in 1999, he co-founded X.com, an online payment company that later merged to form PayPal, which was acquired by eBay in 2002. Musk also beca

ValueError: LLMMathChain._evaluate("
(datetime.today() - datetime(1971, 6, 28)).days
") raised error: Expression (datetime.today() - datetime(1971, 6, 28)).days has forbidden control characters.. Please try again with a valid numerical expression

#### Wikipedia and llm-math tool

In [ ]:
pip install wikipedia

In [ ]:
# install this package: pip install wikipedia

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Let's test it out!
#agent.run("When was Elon musk born? What is his current age in days")
agent.run("How many balloons can we fit in a380?")





> Entering new AgentExecutor chain...
 We need to find the volume of the a380 and the volume of a balloon to determine how many can fit.
Action: Calculator
Action Input: Volume of a380 = 845 m^3, Volume of a balloon = 0.014 m^3
Observation: Answer: 845.014
Thought: This is the total volume of the a380 and the balloon combined.
Action: Calculator
Action Input: Divide 845.014 by 0.014
Observation: Answer: 60358.142857142855
Thought: This is the maximum number of balloons that can fit in the a380.
Final Answer: 60358 balloons can fit in a380.

> Finished chain.


'60358 balloons can fit in a380.'

In [ ]:
pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI

# Using gpt-4
llm_gpt4 = ChatOpenAI(model="gpt-4")

In [ ]:
# pip install -q wikipedia google-search-results langchain langchain-openai

import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentType, initialize_agent, load_tools

# -------------------------------------------------------------------
# 0. API KEYS (set these in your env in practice, not hard-coded)
# -------------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_API_KEY"

# -------------------------------------------------------------------
# 1. LLM: strong reasoning model for tool use
# -------------------------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4.1-mini",   # or "gpt-4.1" / "gpt-4o" if you have access
    temperature=0,          # deterministic for reasoning + tools
)

# -------------------------------------------------------------------
# 2. Tools: SerpAPI + Wikipedia + LLM Math
# -------------------------------------------------------------------
tools = load_tools(
    ["serpapi", "wikipedia", "llm-math"],
    llm=llm,
    serpapi_api_key=os.environ["SERPAPI_API_KEY"],
)

# -------------------------------------------------------------------
# 3. Agent: structured ReAct-style agent for complex, multi-step queries
# -------------------------------------------------------------------
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,   # give some room for multi-step reasoning
    agent_kwargs={
        "system_message": (
            "You are a careful, analytical assistant.\n"
            "- Use SerpAPI for fresh or non-Wikipedia web information.\n"
            "- Use Wikipedia for well-known entities and background facts.\n"
            "- Use llm-math for any non-trivial calculations.\n"
            "For complex questions, break the problem into steps, "
            "use tools when needed, and explain your assumptions clearly."
        )
    },
)

# -------------------------------------------------------------------
# 4. Test query: A380 balloon Fermi estimation
# -------------------------------------------------------------------
query = (
    "How many standard party balloons (30 cm diameter) can fit inside an Airbus A380? "
    "Use realistic assumptions, fetch any needed dimensions, and show the estimation steps."
)

# Recommended (new-style) call:
response = agent.invoke({"input": query})
print(response["output"])

# Legacy style (still works if you prefer):
# answer = agent.run(query)
# print(answer)


ModuleNotFoundError: No module named 'langchain_core.memory'

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentType, initialize_agent, load_tools

# Make sure these are set in your environment in practice
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_API_KEY"

# 1. LLM: switch from old `OpenAI` to chat-based `ChatOpenAI`
llm = ChatOpenAI(
    model="gpt-4.1-mini",   # or "gpt-4.1" / "gpt-4o" if you have access
    temperature=0,          # stable, reasoning-focused
)

# 2. Tools: SerpAPI + Wikipedia + llm-math
tools = load_tools(
    ["serpapi", "wikipedia", "llm-math"],
    llm=llm,
    serpapi_api_key=os.environ.get("SERPAPI_API_KEY")
)

# 3. Agent: use a stronger structured agent for complex queries
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    agent_kwargs={
        "system_message": (
            "You are a careful, up-to-date reasoning assistant.\n"
            "- Use SerpAPI first for current numerical data like per-capita income.\n"
            "- Use Wikipedia for background facts.\n"
            "- Use llm-math for any non-trivial calculations.\n"
            "Always explain your steps briefly and cite the year of any figures you report."
        )
    },
)

# 4. Test it out
query = "What is the per capita income of Dubai? Use a search engine to find th"


ModuleNotFoundError: No module named 'langchain_core.memory'